# Exercise 7: Script transition implications

*Scripts and Encoding · version 0.1.0*

> *In this hypothetical exercise*, a researcher has scraped Kazakh web text in which 60% of the documents are in Cyrillic and 40% in Latin (following the 2021 proposal plus informal variants). Describe three specific problems this mixed-script corpus will cause if used without preprocessing, and propose a concrete solution for each. Would your answer change if the 60/40 split counted words instead of documents?

The code shows two of the problems on a toy version of such a corpus: split vocabularies and document-versus-word shares.

**Setup.** `pip install "turkicnlp==0.5.0"`. Cells that fetch data from the web need internet access.

In [ ]:
from collections import Counter
from turkicnlp.scripts.detector import detect_script
from turkicnlp.scripts.transliterator import Transliterator
from turkicnlp.scripts import Script

corpus = [                                       # toy documents, not real data
    "Қазақстан Республикасы туралы жаңалықтар.",
    "Алматы қаласында жиналыс өтті.",
    "Біз Қазақстан туралы жазамыз және оқимыз.",
    "Qazaqstan Respublikasy turaly jañalyqtar.",
    "Qazaqstan turaly qysqa habar, osy kunderi.",   # informal Latin: dropped diacritics (künderi)
]
scripts = [detect_script(d).name for d in corpus]
print("documents per script:", Counter(scripts))
print("words per script    :", Counter({s: 0 for s in scripts}) + Counter(s for d, s in zip(corpus, scripts) for _ in d.split()))

vocab = Counter(w.strip(".,").lower() for d in corpus for w in d.split())
print("'Kazakhstan' appears as:", {w: n for w, n in vocab.items() if w in ("қазақстан", "qazaqstan")})

to_cyr = Transliterator("kaz", Script.LATIN, Script.CYRILLIC)
print("Latin → Cyrillic view:", [to_cyr.transliterate(d) for d, s in zip(corpus, scripts) if s == "LATIN"])

## Worked solution

1. **Split vocabulary and statistics.** The same word appears as *Қазақстан* and *Qazaqstan*, so counts, subword vocabularies and embeddings treat them as unrelated. *Solution:* keep the original, record the script per document and span, and create a derived view in one script (or train on both with enough data in each), choosing by a task comparison.
2. **Inconsistent and noisy Latin.** Different Latin designs (2017, 2018, 2021) and informal spellings without diacritics (*kunderi* for *künderi*) coexist, so even within the Latin part a word has several forms, and converting such text to Cyrillic produces errors (*кундери*). *Solution:* record the orthography version as metadata, normalise variants of the same design, and apply restoration before or after conversion depending on the model (Exercise 4).
3. **Wrong language labels and duplicates.** Latin Kazakh is easily mislabelled by language identifiers (as Turkish, Uzbek or Karakalpak), and the same news item can appear in both scripts, so deduplication by string misses it and the two copies can end up on both sides of a train/test split. *Solution:* identify the language on the original text, deduplicate on a script-normalised view, and split by source document before creating script variants.

**Documents versus words.** Yes. A 60/40 split by documents can be very different by words if Latin documents are systematically shorter (social media, headlines) or longer; in the toy corpus the shares by documents and by words already differ. Training-data share, vocabulary statistics and tokenisation cost depend on words or tokens, while sampling and deduplication operate on documents, so report both units.